# ACE-Step 1.5 - Google Colab A100

Bu notebook A100 GPU icin hazirlandi. En kaliteli yerel ayar olarak `acestep-v15-xl-sft` DiT modeli ve `acestep-5Hz-lm-4B` LM modeli kullanir.

A100 ile hedef kapasite:
- Maksimum sure: proje limitine gore 600 saniye, yani 10 dakika.
- Rahat baslangic: 120-240 saniye.
- Kalite modu: XL SFT + 4B LM + vLLM backend.
- Hizi arttirmak icin: `acestep-v15-xl-turbo` veya `acestep-v15-turbo` secilebilir.

Colab menusu: `Runtime > Change runtime type > GPU > A100` secili olmali.


In [ ]:
#@title 1) GPU kontrolu test3
!nvidia-smi

import subprocess

gpu_name = subprocess.check_output(
    "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True
).decode().strip()
print("\nAktif GPU:", gpu_name)

if "A100" not in gpu_name:
    print("UYARI: Bu notebook A100 icin optimize edildi. Yine calisir ama ayarlari dusurmen gerekebilir.")

In [ ]:
#@title 2) Repo ve bagimlilik kurulumu
# Ilk calistirmada uzun surebilir. Modeller de ilk uretimde indirilecektir.

%cd /content
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d ACE-Step-1.5 || git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!/root/.local/bin/uv sync
!/root/.local/bin/uv pip install hf_transfer

print("Kurulum tamam.")

In [ ]:
#@title 3) A100 kalite ayarlari ve modelleri hazirla
import os
import shutil
import site
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/ACE-Step-1.5")
DRIVE_ROOT = Path("/content/drive/MyDrive/path")
CHECKPOINTS_DIR = DRIVE_ROOT / "acestep_checkpoints"
OUTPUT_DIR = Path("/content/acestep_outputs")
VENV_DIR = PROJECT_ROOT / ".venv"

def _find_site_packages():
    return next(VENV_DIR.glob("lib/python*/site-packages"), None)

def _ensure_uv_env():
    site_packages = _find_site_packages()
    if site_packages is None:
        print(".venv bulunamadi; uv sync tekrar calistiriliyor...")
        uv_bin = shutil.which("uv") or "/root/.local/bin/uv"
        if not Path(uv_bin).exists() and shutil.which("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
            uv_bin = shutil.which("uv") or "uv"
        subprocess.run([uv_bin, "sync"], cwd=PROJECT_ROOT, check=True)
        site_packages = _find_site_packages()
    if site_packages is None:
        raise RuntimeError(".venv site-packages olusmadi. 2. hucrede uv sync hata verdiyse onun ciktisini kontrol et.")
    site.addsitedir(str(site_packages))
    if str(site_packages) in sys.path:
        sys.path.remove(str(site_packages))
    sys.path.insert(0, str(site_packages))
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    return site_packages

site_packages = _ensure_uv_env()

try:
    import hf_transfer
except ImportError:
    print("hf_transfer venv icinde yok; hizli indirme icin kuruluyor...")
    uv_bin = shutil.which("uv") or "/root/.local/bin/uv"
    subprocess.run([uv_bin, "pip", "install", "hf_transfer"], cwd=PROJECT_ROOT, check=True)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

os.environ["ACESTEP_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["ACESTEP_CHECKPOINTS_DIR"] = str(CHECKPOINTS_DIR)
os.environ["ACESTEP_INIT_LLM"] = "true"
os.environ["ACESTEP_LM_BACKEND"] = "vllm"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Python packages:", site_packages)
print("Drive cache:", DRIVE_ROOT)
print("Models:", CHECKPOINTS_DIR)
print("Outputs:", OUTPUT_DIR)

In [ ]:
#@title 4) Servisi baslat: A100 en iyi kalite
# Ilk defa calistirirken XL DiT ve 4B LM indirir. Bu dosyalar buyuktur.

import os
import shutil
import site
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/ACE-Step-1.5")
DRIVE_ROOT = Path("/content/drive/MyDrive/path")
CHECKPOINTS_DIR = DRIVE_ROOT / "acestep_checkpoints"
OUTPUT_DIR = Path("/content/acestep_outputs")
VENV_DIR = PROJECT_ROOT / ".venv"

def _find_site_packages():
    return next(VENV_DIR.glob("lib/python*/site-packages"), None)

site_packages = _find_site_packages()
if site_packages is None:
    print(".venv bulunamadi; uv sync tekrar calistiriliyor...")
    uv_bin = shutil.which("uv") or "/root/.local/bin/uv"
    if not Path(uv_bin).exists() and shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
        uv_bin = shutil.which("uv") or "uv"
    subprocess.run([uv_bin, "sync"], cwd=PROJECT_ROOT, check=True)
    site_packages = _find_site_packages()
if site_packages is None:
    raise RuntimeError(".venv site-packages olusmadi. 2. hucrede uv sync hata verdiyse onun ciktisini kontrol et.")
site.addsitedir(str(site_packages))
if str(site_packages) in sys.path:
    sys.path.remove(str(site_packages))
sys.path.insert(0, str(site_packages))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import hf_transfer
except ImportError:
    print("hf_transfer venv icinde yok; hizli indirme icin kuruluyor...")
    uv_bin = shutil.which("uv") or "/root/.local/bin/uv"
    subprocess.run([uv_bin, "pip", "install", "hf_transfer"], cwd=PROJECT_ROOT, check=True)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

os.environ["ACESTEP_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["ACESTEP_CHECKPOINTS_DIR"] = str(CHECKPOINTS_DIR)
os.environ["ACESTEP_INIT_LLM"] = "true"
os.environ["ACESTEP_LM_BACKEND"] = "vllm"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.model_downloader import ensure_dit_model, ensure_lm_model, get_checkpoints_dir

DIT_MODEL = "acestep-v15-xl-sft"       # En iyi kalite. Daha hizli: acestep-v15-xl-turbo
LM_MODEL = "acestep-5Hz-lm-4B"         # A100 icin en iyi LM secimi
BACKEND = "vllm"
DEVICE = "cuda"

checkpoints = get_checkpoints_dir(str(CHECKPOINTS_DIR))

print("Modeller su Drive klasorunde aranacak:", checkpoints)
print("Eksik dosya varsa indirilecek; tamamsa tekrar indirilmeyecek.")
print("Indirme sirasi: once DiT tamamen bitecek, sonra LM baslayacak.")

ok, msg = ensure_dit_model(DIT_MODEL, checkpoints, prefer_source="huggingface_only")
print(msg)
if not ok:
    raise RuntimeError(msg)

ok, msg = ensure_lm_model(LM_MODEL, checkpoints, prefer_source="huggingface_only")
print(msg)
if not ok:
    raise RuntimeError(msg)

dit_handler = AceStepHandler()
init_status, can_generate = dit_handler.initialize_service(
    project_root=str(PROJECT_ROOT),
    config_path=DIT_MODEL,
    device=DEVICE,
    compile_model=True,
    offload_to_cpu=False,
    offload_dit_to_cpu=False,
)
print(init_status)
if not can_generate:
    raise RuntimeError("DiT modeli baslatilamadi.")

llm_handler = LLMHandler()
llm_handler.initialize(
    checkpoint_dir=str(CHECKPOINTS_DIR),
    lm_model_path=LM_MODEL,
    backend=BACKEND,
    device=DEVICE,
)

print("ACE-Step A100 servisi hazir.")

In [ ]:
#@title 5) Türkçe Prompt, Lyrics ve Ayarlar (TEST - 90s)
# Türkçe duygusal pop balad, erkek vokal, modern cinematic arrangement

prompt = "Turkish contemporary pop ballad; emotional male lead vocal, clear melodic singing, intimate phrasing, warm piano and strings arrangement, soft electric guitar accents, warm rounded bass, modern radio-ready mix, vocal-forward production, emphasize clear enunciation and natural vibrato, no spoken word, avoid noisy artifacts, sample_rate=44100, vocal_presence=strong, target_duration=90s" #@param {type:"string"}

lyrics = """[Verse 1]\nGece hafif, şehir uyuyor usul,\nKalbim seni sayar, sessiz bir gülüş.\nAdın rüzgâr gibi dolanır içime,\nHer solukta sen, her düşte bir iz.\n\n[Pre-Chorus]\nGözlerimde saklı bir umut var,\nSana dönerken tüm yollar dar.\n\n[Chorus]\nGel, bir dakika yetmez bana,\nSesinle durur zaman, durur dünya.\nEllere gerek yok, yüreğim anlar,\nİçimde sen, hep sen, bir tek anlar.\n\n[Bridge]\nBir nota daha çal, durma lütfen,\nSonsuzluk değil yeter bir izinsin.""" #@param {type:"string"}

# TEST AYARLARI - Hızlı deneme (cızırtı kontrolü için)
duration_seconds = 90 #@param {type:"slider", min:10, max:600, step:10}
vocal_language = "tr" #@param ["tr", "en", "de", "fr", "es", "it", "ja", "ko", "zh", "unknown"]
bpm = 0 #@param {type:"integer"}
keyscale = "" #@param {type:"string"}

# KALITE VE GÜVENLIK AYARLARI
batch_size = 1 #@param {type:"slider", min:1, max:8, step:1}
seed = 12345 #@param {type:"integer"}  # Sabit seed - reproducible test
inference_steps = 60 #@param {type:"slider", min:8, max:100, step:4}  # Yüksek kalite
use_lm_thinking = True #@param {type:"boolean"}
format_prompt_and_lyrics = True #@param {type:"boolean"}
audio_format = "flac" #@param ["flac", "wav", "mp3"]  # FLAC - lossless
use_constrained_decoding = False #@param {type:"boolean"}  # Devre dışı - vokal testi

duration_seconds = max(10, min(600, int(duration_seconds)))
bpm_value = None if int(bpm) <= 0 else int(bpm)
lyrics_value = lyrics.strip() if lyrics.strip() else "[Instrumental]"

print("=" * 60)
print("TEST AYARLARI - 90s Türkçe Duygusal Pop")
print("=" * 60)
print(f"Prompt: {prompt[:60]}...")
print(f"Lyrics: {len(lyrics_value)} karakter")
print(f"Sure: {duration_seconds} saniye")
print(f"Dil: {vocal_language}")
print(f"Seed: {seed} (reproducible)")
print(f"Inference Steps: {inference_steps} (yüksek kalite)")
print(f"Format: {audio_format} (lossless)")
print(f"Vokal kontrolü: use_constrained_decoding={use_constrained_decoding}")
print("=" * 60)

In [ ]:
#@title 6) TURK TEST - 90s Müzik Üret & Analiz Yap
from IPython.display import Audio, display
from acestep.inference import GenerationParams, GenerationConfig, format_sample, generate_music
import torch
import numpy as np

print("🎵 TEST ÜRETIM BAŞLIYOR: 90s Türkçe Duygusal Pop")
print("-" * 60)

caption_value = prompt
keyscale_value = keyscale.strip()
use_cot_metas = True

# LLM ile prompt/lyrics formatla (varsa)
if format_prompt_and_lyrics and llm_handler is not None:
    print("📝 LLM ile prompt/lyrics formatlanıyor...")
    formatted = format_sample(
        llm_handler=llm_handler,
        caption=prompt,
        lyrics=lyrics_value,
        user_metadata={"duration": duration_seconds, "language": vocal_language},
    )
    if formatted.success:
        caption_value = formatted.caption or caption_value
        lyrics_value = formatted.lyrics or lyrics_value
        bpm_value = bpm_value or formatted.bpm
        keyscale_value = keyscale_value or (formatted.keyscale or "")
        if formatted.duration:
            duration_seconds = int(min(600, max(10, float(formatted.duration))))
        use_cot_metas = False
        print("✅ LLM formatlaması başarılı")
    else:
        print(f"⚠️ LLM formatlaması atlandı: {formatted.error}")

# Üretim parametreleri
params = GenerationParams(
    task_type="text2music",
    caption=caption_value,
    lyrics=lyrics_value,
    duration=duration_seconds,
    bpm=bpm_value,
    keyscale=keyscale_value,
    vocal_language=vocal_language,
    inference_steps=inference_steps,
    seed=seed,
    thinking=use_lm_thinking,
    use_cot_metas=use_cot_metas,
    use_cot_caption=use_lm_thinking,
    use_cot_language=use_lm_thinking,
    use_constrained_decoding=use_constrained_decoding,
    enable_normalization=True,
    normalization_db=-1.0,
)

# Üretim konfigurasyonu
config = GenerationConfig(
    batch_size=batch_size,
    audio_format=audio_format,
    use_random_seed=(seed < 0),
    allow_lm_batch=True,
    lm_batch_chunk_size=8,
)

print(f"🎹 Üretim yapılıyor: duration={duration_seconds}s, inference_steps={inference_steps}, seed={seed}...")
result = generate_music(dit_handler, llm_handler, params, config, save_dir=str(OUTPUT_DIR))

if not result.success:
    print(f"❌ HATA: {result.error}")
    raise RuntimeError(result.error)

print("\n" + "=" * 60)
print("✅ ÜRETIM TAMAM")
print("=" * 60)

# Audio analizi
for i, audio_dict in enumerate(result.audios):
    path = audio_dict.get("path") or audio_dict.get("audio_path")
    tensor = audio_dict.get("tensor")
    
    print(f"\n📊 Audio {i+1} Analizi:")
    print(f"  Path: {path}")
    print(f"  Sample Rate: {audio_dict.get('sample_rate', 48000)} Hz")
    
    if tensor is not None:
        tensor_np = tensor.cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor
        peak = float(np.abs(tensor_np).max())
        mean = float(np.abs(tensor_np).mean())
        rms = float(np.sqrt(np.mean(tensor_np**2)))
        
        has_nan = np.isnan(tensor_np).any()
        has_inf = np.isinf(tensor_np).any()
        
        print(f"  Şekil: {tensor_np.shape}")
        print(f"  Peak: {peak:.6f} {'⚠️ CLIPPING!' if peak > 1.0 else '✓'}")
        print(f"  RMS: {rms:.6f}")
        print(f"  Ortalama: {mean:.6f}")
        print(f"  NaN: {has_nan} {'❌' if has_nan else '✓'}")
        print(f"  Inf: {has_inf} {'❌' if has_inf else '✓'}")
        
        if peak > 1.0:
            print(f"  ⚠️ DİKKAT: Peak > 1.0 — clipping riski!")
    
    if path and os.path.exists(path):
        print(f"  Dosya boyutu: {os.path.getsize(path) / (1024*1024):.2f} MB")
        print(f"  ▶️ Dinle:")
        display(Audio(path))
    else:
        print(f"  ❌ Dosya bulunamadı")

print("\n" + "=" * 60)
print("📝 Özet:")
print(f"  Toplam audio: {len(result.audios)}")
print(f"  Status: {result.status_message[:100]}")
print("=" * 60)

In [ ]:
#@title Opsiyonel: Gradio arayuzunu public link ile ac
# Python API yerine web arayuzu kullanmak istersen bu hucreyi calistir.
# Durdurmak icin Colab runtime'i interrupt edebilirsin.

%cd /content/ACE-Step-1.5
from google.colab import drive
from pathlib import Path
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
!/root/.local/bin/uv pip install hf_transfer
!ACESTEP_CHECKPOINTS_DIR=/content/drive/MyDrive/path/acestep_checkpoints HF_HUB_ENABLE_HF_TRANSFER=1 /root/.local/bin/uv run acestep --share --server-name 0.0.0.0 --init_service true --config_path acestep-v15-xl-sft --lm_model_path acestep-5Hz-lm-4B --init_llm true